# ProteinMPNN Inverse Folding

This notebook demonstrates the three ProteinMPNN tools. First, we use `run_proteinmpnn_sample` to generate new amino acid sequences conditioned on a target backbone structure. Second, we use `run_proteinmpnn_score` to evaluate how well an existing sequence fits a given structure by computing structure-conditioned likelihoods. Third, we use `run_proteinmpnn_gradient` to differentiate ProteinMPNN's mean-NLL objective with respect to a continuous `(L, 20)` distribution — useful as a structure-conditioned loss inside MCMC or gradient descent over relaxed sequences.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("proteinmpnn")
display_overview("proteinmpnn")
display_docs_section("proteinmpnn", "Background")

# ProteinMPNN

ProteinMPNN is a deep learning model for protein sequence design given a protein backbone structure ("inverse folding"). It uses message passing neural networks to predict amino acid sequences that will fold into a target 3D structure. This module provides interfaces for *Sequence Sampling* (generating new sequences for a given backbone), *Sequence Scoring* (evaluating how well a sequence fits a structure), and *Relaxed-Sequence Gradients* (differentiating ProteinMPNN's perplexity objective with respect to a continuous `(L, 20)` distribution for use as a structure-conditioned loss in MCMC or gradient descent).

## Background

**What does this tool do?**
ProteinMPNN solves the "[inverse folding](https://en.wikipedia.org/wiki/Protein_design#Inverse_folding)" problem: given a protein backbone structure (the 3D coordinates of N, CA, C, O atoms), predict which amino acid sequence will fold into that structure. This is the inverse of structure prediction.

**Why is this important?**
Inverse folding enables:
- Protein engineering: Redesign natural proteins with improved stability, solubility, or expression.
- Scaffold design: Create proteins with desired shapes for binding, catalysis, or assembly.
- Therapeutic development: Design antibodies, enzymes, and binding proteins.
- Experimental validation: Generate multiple sequence candidates for a single structure to find the best experimental hits.

**Scientific foundation:**
ProteinMPNN uses a [message passing neural network](https://en.wikipedia.org/wiki/Graph_neural_network) architecture:

1. **Graph representation**: The protein backbone is represented as a graph where each residue is a node and edges connect spatially close residues (within ~30A).
2. **Message passing**: Information flows between connected residues over multiple rounds, allowing the model to learn long-range dependencies.
3. **[Autoregressive](https://en.wikipedia.org/wiki/Autoregressive_model) decoding**: Sequences are generated one residue at a time, conditioned on previously generated residues and the full structural context.
4. **Training**: The model was trained on ~19,000 protein structures from the [PDB](https://www.rcsb.org/) to maximize the probability of native sequences given their structures.

## Available tools

In [2]:
display_available_tools("proteinmpnn")

- **`run_proteinmpnn_gradient()`** — Compute ProteinMPNN structure-conditioned perplexity gradient for relaxed protein sequences
- **`run_proteinmpnn_sample()`** — Sample protein sequences using ProteinMPNN
- **`run_proteinmpnn_score()`** — Score protein sequences using ProteinMPNN

### `run_proteinmpnn_sample`

ProteinMPNN designs new protein sequences that are predicted to fold into a target backbone structure. We use Green Fluorescent Protein (GFP) as our target, a 238-residue beta-barrel protein whose well-characterized fold makes it an ideal test case for inverse folding. ProteinMPNN analyzes the 3D backbone coordinates and autoregressively generates amino acid sequences optimized for the target shape. The model also supports antibody-optimized weights via the `abmpnn` model choice.

In [3]:
from proto_tools import (
    InverseFoldingStructureInput,
    run_proteinmpnn_sample,
    ProteinMPNNSampleInput,
    ProteinMPNNSampleConfig,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [4]:
# Display input docs
display_api_reference("proteinmpnn", "input", "run_proteinmpnn_sample")

# Load the GFP structure and wrap it in an inverse folding input
gfp_structure = get_gfp_structure()
struct_input = InverseFoldingStructureInput(structure=gfp_structure, chains_to_redesign=["A"])
inputs = ProteinMPNNSampleInput(inputs=[struct_input])

**Input** — `InverseFoldingInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `list[proto_tools.tools.inverse_folding.shared_data_models.InverseFoldingStructureInput]` | required | Per-structure inputs, each containing a structure and optional selections. |

In [5]:
# Display config docs
display_api_reference("proteinmpnn", "config", "run_proteinmpnn_sample")

# Configure sampling | see docs above for all fields
config = ProteinMPNNSampleConfig(
    num_sequences_per_structure=2,  # Generate 2 sequence designs
    temperature=0.1,                # Conservative designs (closer to consensus)
    device="cuda",                  # Change to "cpu" if no GPU available
)

**Config** — `ProteinMPNNSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on. Options include 'cuda' (NVIDIA GPU), 'cpu' (CPU execution) |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `num_sequences_per_structure` | `int` | `1` | Total number of sequences to generate per input structure. |
| `batch_size` | `int | None` | `None` | Number of sequences to process simultaneously on GPU. Defaults to num_sequences_per_structure. |
| `temperature` | `float` | `0.1` | Sampling temperature; lower = greedier, higher = more diverse |
| `model_choice` | `Literal['proteinmpnn', 'v_48_002', 'v_48_010', 'v_48_030', 'abmpnn', 'soluble']` | `'proteinmpnn'` | Weights: proteinmpnn (=v_48_020), v_48_{002,010,030} noise variants, abmpnn, soluble |
| `backbone_noise` | `float` | `0.0` | Gaussian noise (A) on backbone coords; raise (e.g. 0.02) for diversity |
| `excluded_amino_acids` | `list[Literal['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']] | None` | `None` | Single-letter codes of amino acids to exclude (e.g. ['C'] to forbid cysteine) |

In [6]:
# Run the sampling function
result = run_proteinmpnn_sample(inputs, config)

Running run_proteinmpnn_sample [00:00]

In [7]:
# Display output docs
display_api_reference("proteinmpnn", "output", "run_proteinmpnn_sample")

# Print the designed sequences
for i, seq_res in enumerate(result.designed_sequences):
    print(f"Structure {i} designs:")
    for j, seq in enumerate(seq_res.sequences, 1):
        preview = f"{seq[:50]}..." if len(seq) > 50 else seq
        print(f"  Design {j}: {preview}")
        print(f"            Length: {len(seq)} residues")

**Output** — `InverseFoldingOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `designed_sequences` | `list[Annotated[proto_tools.tools.inverse_folding.shared_data_models.DesignedSequences, SerializeAsAny()]]` | required | List of sequences predicted for the input structures |

Structure 0 designs:
  Design 1: SAGAALFTGVVPIKVRLDGDVNGHKFSVEGEGEGDATTGKLELKFVCTTG...
            Length: 227 residues
  Design 2: SAGAELFKGVVPIKVELDGDVNGKKFSVVGEGEGDATTGTLNLKFVCTTG...
            Length: 227 residues


### `run_proteinmpnn_score`

The scoring function evaluates how well a protein sequence fits a given backbone structure. It computes per-residue log-likelihoods using ProteinMPNN's structure-conditioned language model, producing metrics such as perplexity and average log-likelihood. Lower perplexity indicates better sequence-structure compatibility, making this useful for ranking candidate designs or evaluating natural sequences against their native structures.

Here we score one of the designed sequences from the sampling step above to evaluate how well it fits the GFP backbone.

In [8]:
from proto_tools import (
    ProteinMPNNScoringInput,
    ProteinMPNNScoringConfig,
    SequenceStructurePair,
    run_proteinmpnn_score,
)

In [9]:
# Display input docs
display_api_reference("proteinmpnn", "input", "run_proteinmpnn_score")

# Score the first designed sequence against the GFP backbone
designed_seq = result.designed_sequences[0].sequences[0]
pair = SequenceStructurePair(sequence=designed_seq, structure=gfp_structure)
scoring_input = ProteinMPNNScoringInput(sequence_structure_pairs=[pair])

**Input** — `ProteinMPNNScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequence_structure_pairs` | `list[proto_tools.tools.inverse_folding.shared_data_models.SequenceStructurePair]` | required | List of sequence-structure pairs to score |

In [10]:
# Display config docs
display_api_reference("proteinmpnn", "config", "run_proteinmpnn_score")

# Configure scoring | see docs above for all fields
scoring_config = ProteinMPNNScoringConfig(
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ProteinMPNNScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on. Options include 'cuda' (NVIDIA GPU), 'cpu' (CPU execution) |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `fixed_positions` | `dict[str, list[int]] | None` | `None` | Dictionary mapping chain IDs to fixed positions excluded from scoring metrics |
| `return_logits` | `bool` | `False` | Whether to include per-position logits in the output. Disable to save memory. |
| `model_choice` | `Literal['proteinmpnn', 'v_48_002', 'v_48_010', 'v_48_030', 'abmpnn', 'soluble']` | `'proteinmpnn'` | Weights: proteinmpnn (=v_48_020), v_48_{002,010,030} noise variants, abmpnn, soluble |

In [11]:
# Run the scoring function
score_result = run_proteinmpnn_score(scoring_input, scoring_config)

Running run_proteinmpnn_score [00:00]

In [12]:
# Display output docs
display_api_reference("proteinmpnn", "output", "run_proteinmpnn_score")

# Display scoring metrics for the designed sequence
score = score_result.scores[0]
print(f"Perplexity: {score['perplexity']:.3f}")
print(f"Log-likelihood: {score['log_likelihood']:.3f}")
print(f"Avg log-likelihood: {score['avg_log_likelihood']:.3f}")

**Output** — `InverseFoldingScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `list[proto_tools.tools.inverse_folding.shared_data_models.InverseFoldingScoringMetrics]` | required | List of scoring outputs, one per input sequence-structure pair |

Perplexity: 2.027
Log-likelihood: -160.404
Avg log-likelihood: -0.707


### `run_proteinmpnn_gradient`

The gradient tool takes a relaxed `(L, 20)` distribution over amino acids together with a backbone structure and returns the gradient of ProteinMPNN's structure-conditioned mean-NLL objective with respect to those input logits. Use this as a differentiable, structure-conditioned loss inside MCMC or gradient descent over relaxed protein sequences. Set `compute_gradient=False` to get the same scalar objective without the backward pass — useful for scoring proposals.

In [13]:
from proto_tools import (
    ProteinMPNNGradientConfig,
    ProteinMPNNGradientInput,
    run_proteinmpnn_gradient,
)
from proto_tools.utils import one_hot_protein_logits

In [14]:
# Display input docs
display_api_reference("proteinmpnn", "input", "run_proteinmpnn_gradient")

# Seed a relaxed distribution from the designed sequence we sampled earlier
# (sharpness=2.0 yields a biased-but-not-saturated softmax target).
logits = one_hot_protein_logits(designed_seq, sharpness=2.0)
gradient_input = ProteinMPNNGradientInput(
    logits=logits,
    structure=gfp_structure,
    chains_to_redesign=["A"],
)

**Input** — `ProteinMPNNGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `logits` | `list[list[float]]` | required | Relaxed sequence logits with shape (L, 20) in canonical amino-acid order. |
| `structure` | `proto_tools.entities.structures.structure.Structure` | required | Backbone structure for ProteinMPNN conditioning. |
| `chains_to_redesign` | `proto_tools.entities.structures.selection.ChainSelection | None` | `None` | Structure chains to score. Defaults to all chains in the structure. |
| `fixed_positions` | `proto_tools.entities.structures.selection.ResidueSelection | None` | `None` | Per-chain 1-indexed residue positions excluded from the ProteinMPNN objective. |
| `temperature` | `float | None` | `None` | Softmax temperature. Applies softmax(input / T) when set. |

In [15]:
# Display config docs
display_api_reference("proteinmpnn", "config", "run_proteinmpnn_gradient")

# Configure the gradient tool | see docs above for all fields
gradient_config = ProteinMPNNGradientConfig(
    model_choice="proteinmpnn",
    use_ste=True,            # Hard one-hot forward, soft-probability backward
    compute_gradient=True,   # Set False for forward-only NLL scoring
    device="cuda",           # Change to "cpu" if no GPU available
)

**Config** — `ProteinMPNNGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `model_choice` | `Literal['proteinmpnn', 'v_48_002', 'v_48_010', 'v_48_030', 'abmpnn', 'soluble']` | `'proteinmpnn'` | Weights: proteinmpnn (=v_48_020), v_48_{002,010,030} noise variants, abmpnn, soluble |
| `use_ste` | `bool` | `True` | Hard one-hot forward pass with soft-probability gradients. |
| `compute_gradient` | `bool` | `True` | Run backward pass and return gradient; set False for forward-only log-likelihood. |

In [16]:
# Run the gradient function
gradient_result = run_proteinmpnn_gradient(gradient_input, gradient_config)

# Display output docs
display_api_reference("proteinmpnn", "output", "run_proteinmpnn_gradient")

# Inspect the scalar objective and gradient shape
print(f"Mean NLL:           {gradient_result.loss:.3f}")
print(f"Perplexity:         {gradient_result.metrics['perplexity']:.3f}")
print(f"Avg log-likelihood: {gradient_result.metrics['avg_log_likelihood']:.3f}")
assert gradient_result.gradient is not None  # backward pass enabled
print(f"Gradient shape:     ({len(gradient_result.gradient)}, {len(gradient_result.gradient[0])})")
print(f"Gradient row 0 (first 5 cols): {[round(v, 4) for v in gradient_result.gradient[0][:5]]}")

Running run_proteinmpnn_gradient [00:00]

**Output** — `ProteinMPNNGradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `gradient` | `list[list[float]] | None` | `None` | Gradient w.r.t. input logits. None when compute_gradient=False. |
| `loss` | `float` | required | Scalar objective value returned by the backend for this relaxed sequence. |
| `metrics` | `dict[str, Any]` | `{}` | Auxiliary metrics reported alongside the scalar objective value. |
| `vocab` | `list[str]` | required | Amino-acid symbols defining the column ordering for both the input logits and the returned gradient. |

Mean NLL:           0.729
Perplexity:         2.072
Avg log-likelihood: -0.729
Gradient shape:     (227, 20)
Gradient row 0 (first 5 cols): [-0.0027, 0.0027, -0.0014, 0.0, 0.0024]


## Export Results

Designed sequences can be saved to standard file formats for downstream analysis, gene synthesis, or further computational evaluation such as structure prediction or molecular dynamics.

In [17]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="proteinmpnn_designs", export_path=output_dir, file_format="fasta")
print(f"Sampling results exported to {output_dir / 'proteinmpnn_designs.fasta'}")

score_result.export(name="proteinmpnn_scores", export_path=output_dir, file_format="csv")
print(f"Scoring results exported to {output_dir / 'proteinmpnn_scores.csv'}")

Sampling results exported to example_output/proteinmpnn_designs.fasta
Scoring results exported to example_output/proteinmpnn_scores.csv
